# 📖 Eksplorasi Dataset Tarteel EveryAyah & Model Whisper Quran
### Automatic Speech Recognition (ASR) untuk Al-Qur'an & Pelafalan Huruf - Qurava AI

Notebook ini mendemonstrasikan:
1. **Memuat Dataset Audio Quran**: `tarteel-ai/everyayah` dari Hugging Face.
2. **Eksplorasi Fitur Audio**: Memeriksa gelombang suara (*waveform*), frekuensi sampling, qari, serta transkripsi teks Arab.
3. **Memutar Audio di Notebook**: Menggunakan pemutar audio interaktif `IPython.display.Audio`.
4. **Pengujian Transkripsi AI**: Menguji model **`tarteel-ai/whisper-base-ar-quran`** untuk mengenali audio bacaan dan mencocokkannya dengan teks ayat asli (*Ground Truth vs Prediction*).


## 1. Persiapan Lingkungan & Pustaka Dependensi
Jika belum terpasang, jalankan perintah instalasi berikut di terminal atau sel notebook ini:

```bash
pip install datasets transformers torch torchaudio soundfile librosa
```


In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import IPython.display as ipd

from datasets import load_dataset
from transformers import pipeline

print('✅ Semua library berhasil di-import!')


c:\Users\Arif\Github\lomba\qurava-unej\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


: 

## 2. Memuat Dataset `tarteel-ai/everyayah`

> 💡 **Tips Praktis (Streaming Mode):**
> Dataset `tarteel-ai/everyayah` berisi puluhan ribu rekaman audio dari berbagai qari dunia dengan ukuran total puluhan gigabyte.
> Dengan menggunakan opsi **`streaming=True`**, kita dapat langsung mengakses dan memutar sampel audio **seketika** tanpa harus menunggu unduhan gigabyte data ke penyimpanan lokal.


In [ ]:
print('⏳ Menghubungkan ke Hugging Face Dataset: tarteel-ai/everyayah (Streaming Mode)...')

# Memuat dataset secara streaming agar hemat kuota dan memori disk
ds = load_dataset('tarteel-ai/everyayah', split='train', streaming=True)

print('✅ Berhasil terhubung ke dataset EveryAyah!')


## 3. Eksplorasi Metadata & Sampel Pertama
Mari kita ambil sampel rekaman pertama dan melihat data apa saja yang tersimpan di dalamnya.


In [ ]:
# Ambil sampel pertama menggunakan iterator
dataset_iter = iter(ds)
sample = next(dataset_iter)

print('=== INFORMASI SAMPEL AUDIO ===')
print(f'Surah Index     : {sample.get("surah", sample.get("surah_number", "N/A"))}')
print(f'Ayah Index      : {sample.get("ayah", sample.get("ayah_number", "N/A"))}')
print(f'Nama Qari       : {sample.get("reciter", sample.get("reciter_name", "N/A"))}')
print(f'Teks Arab       : {sample.get("text", sample.get("ayah_text", "N/A"))}')
print(f'Sample Rate     : {sample["audio"]["sampling_rate"]} Hz')
print(f'Total Titik Data: {len(sample["audio"]["array"]):,} sinyal audio')
durasi = len(sample["audio"]["array"]) / sample["audio"]["sampling_rate"]
print(f'Estimasi Durasi : {durasi:.2f} detik')


## 4. Memutar Audio Rekaman Bacaan
Jalankan sel berikut untuk mendengarkan langsung rekaman suara qari melalui pemutar audio interaktif:


In [ ]:
audio_array = sample['audio']['array']
sr = sample['audio']['sampling_rate']

# Visualisasi Gelombang Suara (Waveform)
plt.figure(figsize=(12, 3), dpi=100)
time_axis = np.linspace(0, len(audio_array) / sr, num=len(audio_array))
plt.plot(time_axis, audio_array, color='#1885A4', linewidth=0.6)
plt.title(f'Waveform Audio Bacaan Al-Qur\'an ({sample.get("reciter", "Qari")})', fontsize=11, fontweight='bold')
plt.xlabel('Waktu (detik)', fontsize=10)
plt.ylabel('Amplitudo', fontsize=10)
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

# Pemutar Audio Interaktif
ipd.Audio(audio_array, rate=sr)


## 5. Memuat Model AI: `tarteel-ai/whisper-base-ar-quran`
Sekarang kita siapkan model Speech-to-Text Whisper yang telah di-*fine-tune* oleh Tarteel AI untuk mengenali teks Al-Qur'an.


In [ ]:
print('⏳ Mengunduh & memuat model tarteel-ai/whisper-base-ar-quran...')

# pipeline ASR otomatis
# device=-1 untuk CPU, device=0 untuk GPU NVIDIA CUDA
asr_pipe = pipeline(
    'automatic-speech-recognition',
    model='tarteel-ai/whisper-base-ar-quran',
    device=-1
)

print('✅ Model Tarteel Whisper siap digunakan!')


## 6. Uji Transkripsi: Membandingkan Teks Asli vs Prediksi AI
Kita masukkan sinyal audio dari dataset EveryAyah ke model Whisper untuk melihat apakah model mampu menebak teks ayat Al-Qur'an dengan benar.


In [ ]:
print('⏳ Mentranskripsi audio rekaman...')

# Whisper menerima input dictionary audio {'array': ..., 'sampling_rate': ...}
prediction = asr_pipe({
    'array': sample['audio']['array'],
    'sampling_rate': sample['audio']['sampling_rate']
})

ground_truth = sample.get('text', sample.get('ayah_text', ''))
predicted_text = prediction['text']

print('=' * 60)
print('HASIL EVALUASI SPEECH RECOGNITION (ASR):')
print('=' * 60)
print(f'📖 Teks Asli (Ground Truth) : {ground_truth}')
print(f'🤖 Prediksi Whisper AI      : {predicted_text}')
print('=' * 60)


## 7. Pengujian Batch (5 Sampel Audio Berturut-turut)
Menguji performa transkripsi pada beberapa ayat berurutan untuk melihat konsistensi akurasi model Whisper.


In [ ]:
results = []
print('⏳ Menjalankan pengujian pada 5 sampel audio...')

for i in range(5):
    current_sample = next(dataset_iter)
    pred = asr_pipe({
        'array': current_sample['audio']['array'],
        'sampling_rate': current_sample['audio']['sampling_rate']
    })
    
    results.append({
        'No': i + 1,
        'Surah/Ayah': f"{current_sample.get('surah', '')}:{current_sample.get('ayah', '')}",
        'Teks Asli': current_sample.get('text', current_sample.get('ayah_text', '')),
        'Prediksi AI': pred['text']
    })

df_results = pd.DataFrame(results)
display(df_results)


## 8. Penerapan pada Proyek Qurava

Teknik transkripsi di atas adalah mekanisme yang sama dengan fitur verifikasi suara di aplikasi web **Qurava**:

1. **Pada Latihan Huruf Hijaiyah** ([`src/app/iqra/hijaiyah/[id]/page.tsx`](file:///c:/Users/Arif/Github/lomba/qurava-unej/src/app/iqra/hijaiyah/[id]/page.tsx)):
   - Suara pengguna merekam pengucapan huruf (misal: *"Alif"* atau *"Baa"*).
   - Whisper menghasilkan teks aksara Arab (`"الف"` atau `"باء"`).
   - Web mencocokkannya dengan kamus alias huruf Hijaiyah.

2. **Pada Latihan Membaca Ayat** ([`src/app/ayat/[id]/page.tsx`](file:///c:/Users/Arif/Github/lomba/qurava-unej/src/app/ayat/[id]/page.tsx)):
   - Pengguna membaca ayat Al-Fatihah.
   - Whisper mentranskripsi bacaan dan sistem memvalidasi kebenaran lafadznya terhadap teks ayat Al-Qur'an resmi.
